# Enerji Tüketimi Tahminlemesi (Haftalık, Aylık, Yıllık)

Bu notebook, derste işlenen derin öğrenme ve zaman serisi (Time Series) analiz yöntemlerini kullanarak enerji tüketimi (PJME_hourly) verisi üzerinde LSTM tabanlı modeller ile **Haftalık (7 gün)**, **Aylık (30 gün)** ve **Yıllık (365 gün)** tahminlemeler yapar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import os

import warnings
warnings.filterwarnings('ignore')

## 1. Veri Yükleme ve Ön İşleme
Verimiz saatlik frekansta olduğu için, haftalık, aylık ve yıllık makro tahminler yapabilmek adına veriyi öncelikle **günlük (daily) ortalamalara** indirgiyoruz.

In [ ]:
file_path = 'archive/PJME_hourly.csv'
df = pd.read_csv(file_path)
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.set_index('Datetime')
df = df.sort_index()

# Günlük ortalamaya çeviriyoruz
df_daily = df.resample('D').mean()
df_daily = df_daily.dropna()

plt.figure(figsize=(14, 5))
plt.plot(df_daily.index, df_daily['PJME_MW'])
plt.title('PJME Günlük Enerji Tüketimi')
plt.xlabel('Tarih')
plt.ylabel('MW')
plt.show()

## 2. LSTM İçin Veri Hazırlığı ve Model Fonksiyonları
Zaman serisi verisini LSTM modeline verebilmek için (Gözlem Sayısı, Zaman Adımı, Özellik Sayısı) boyutlarına getirmemiz ve ölçeklendirmemiz (Scaling) gerekiyor.

In [ ]:
def create_sequences(data, look_back, forecast_horizon):
    X, y = [], []
    for i in range(len(data) - look_back - forecast_horizon + 1):
        X.append(data[i:(i + look_back)])
        y.append(data[(i + look_back):(i + look_back + forecast_horizon)])
    return np.array(X), np.array(y)

def build_lstm_model(look_back, forecast_horizon):
    model = Sequential()
    # Derste işlenen derin öğrenme mimarisine uygun LSTM
    model.add(LSTM(64, activation='relu', return_sequences=True, input_shape=(look_back, 1)))
    model.add(Dropout(0.2))
    model.add(LSTM(32, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(forecast_horizon))
    model.compile(optimizer='adam', loss='mse')
    return model

def train_and_predict(df_daily, look_back, forecast_horizon, title, epochs=10, batch_size=32):
    print(f"--- {title} Tahmini İçin Model Eğitiliyor ---")
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(df_daily)
    
    X, y = create_sequences(scaled_data, look_back, forecast_horizon)
    
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    model = build_lstm_model(look_back, forecast_horizon)
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_test, y_test), verbose=1)
    
    # Gelecek Tahmini
    last_sequence = scaled_data[-look_back:]
    last_sequence = last_sequence.reshape((1, look_back, 1))
    
    future_pred_scaled = model.predict(last_sequence)
    future_pred = scaler.inverse_transform(future_pred_scaled)[0]
    
    # Görselleştirme
    plt.figure(figsize=(10, 5))
    last_actual_days = df_daily.iloc[-look_back:].index
    future_dates = pd.date_range(start=last_actual_days[-1] + pd.Timedelta(days=1), periods=forecast_horizon)
    
    plt.plot(last_actual_days, df_daily.iloc[-look_back:].values, label='Gerçek Veri (Son Günler)')
    plt.plot(future_dates, future_pred, label=f'Gelecek {forecast_horizon} Gün Tahmini', marker='o')
    plt.title(f'Enerji Tüketimi {title} Tahmini (LSTM)')
    plt.xlabel('Tarih')
    plt.ylabel('Megawatt (MW)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Eğitim Loss Grafiği
    plt.figure(figsize=(6, 4))
    plt.plot(history.history['loss'], label='Eğitim Kaybı')
    plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
    plt.title(f'{title} Modeli Kayıp Grafiği')
    plt.legend()
    plt.show()

## 3. Haftalık Tahmin
Son 60 güne bakarak **gelecek 7 günün** tahmini yapılır.

In [ ]:
train_and_predict(df_daily, look_back=60, forecast_horizon=7, title="Haftalık", epochs=10)

## 4. Aylık Tahmin
Son 90 güne bakarak **gelecek 30 günün** tahmini yapılır.

In [ ]:
train_and_predict(df_daily, look_back=90, forecast_horizon=30, title="Aylık", epochs=10)

## 5. Yıllık Tahmin
Son 365 güne bakarak **gelecek 365 günün** tahmini yapılır. Yıllık veri çok uzun olduğu için eğitim biraz daha sürebilir.

In [ ]:
train_and_predict(df_daily, look_back=365, forecast_horizon=365, title="Yıllık", epochs=10)